# Exploring tiny_glove Word Vectors

---

# Step 1: Import Libraries

In [1]:
import json
import numpy as np
from numpy.linalg import norm

---

# Step 2: Load tiny_glove

In [2]:
with open("./datasets/tiny_glove.json", "r") as f:
    glove = json.load(f)

In [3]:
print("Vocabulary size:", len(glove))

sample_words = list(glove.keys())[:20]

print("\nFirst 20 words:")
print(sample_words)

Vocabulary size: 4993

First 20 words:
['development', 'episodes', 'bottom', 'elsewhere', 'usa', 'chase', 'via', 'windows', 'sunni', 'pakistan', 'theory', 'towns', 'arafat', 'excellent', 'role', 'almost', 'closing', 'laboratory', 'elderly', 'inter']


---

# Step 3: Inspect One Word Vector

In [4]:
word = "king"

vector = np.array(glove[word])

In [5]:
print("Word:", word)

print("Vector shape:", vector.shape)

print("First 10 values:")
print(vector[:10])

Word: king
Vector shape: (50,)
First 10 values:
[ 0.50450999  0.68607002 -0.59517002 -0.022801    0.60045999 -0.13497999
 -0.08813     0.47376999 -0.61798    -0.31011999]


---

# Step 4: Build Utility Functions

In [6]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))


def get_vector(word):
    if word in glove:
        return np.array(glove[word])
    return None

---

# Step 5: Compare Basic Similarities

In [28]:
pairs = [
    ("king", "queen"),
    ("man", "woman"),
    ("doctor", "nurse"),
    ("king", "apple"),
    ("teacher", "rich"),
    ("usa", "rich")
]

In [29]:
for w1, w2 in pairs:
    v1 = get_vector(w1)
    v2 = get_vector(w2)

    if v1 is not None and v2 is not None:
        sim = cosine_similarity(v1, v2)

        print(f"{w1:10s} vs {w2:10s} -> {sim:.4f}")
    else:
        print(f"Missing word: {w1} or {w2}")

king       vs queen      -> 0.7839
man        vs woman      -> 0.8860
doctor     vs nurse      -> 0.7977
king       vs apple      -> 0.3047
teacher    vs rich       -> 0.3257
usa        vs rich       -> 0.2253


---

# Step 6: Find Nearest Words

In [9]:
def nearest_words(target_word, top_n=10):
    if target_word not in glove:
        return []

    target_vec = get_vector(target_word)

    scores = []

    for word in glove:
        if word == target_word:
            continue

        sim = cosine_similarity(target_vec, get_vector(word))

        scores.append((word, sim))

    scores.sort(key=lambda x: x[1], reverse=True)

    return scores[:top_n]

In [31]:
target = "china"

neighbors = nearest_words(target, top_n=10)

print("Nearest words to:", target)

for word, score in neighbors:
    print(f"{word:15s} {score:.4f}")

Nearest words to: china
taiwan          0.9361
chinese         0.8957
beijing         0.8921
mainland        0.8645
japan           0.8429
vietnam         0.8287
korea           0.8235
hong            0.8147
kong            0.8008
asian           0.7915


---

# Step 7: Compare Multiple Professions

In [11]:
profession_words = [
    "doctor",
    "nurse",
    "engineer",
    "teacher"
]

In [12]:
for base_word in profession_words:
    print(f"\nNearest to {base_word}:")
    
    neighbors = nearest_words(base_word, top_n=5)

    for word, score in neighbors:
        print(f"{word:15s} {score:.4f}")


Nearest to doctor:
nurse           0.7977
patient         0.7612
child           0.7559
teacher         0.7538
doctors         0.7394

Nearest to nurse:
doctor          0.7977
child           0.7341
teacher         0.7242
patient         0.7242
parents         0.7182

Nearest to engineer:
engineers       0.7153
worked          0.7083
retired         0.6979
engineering     0.6914
pilot           0.6813

Nearest to teacher:
student         0.8962
graduate        0.8133
teaching        0.8129
taught          0.8076
school          0.7855


---

# Step 8: Word Arithmetic

We test:

$king - man + woman \approx queen$

In [13]:
result_vector = (
    get_vector("king")
    - get_vector("man")
    + get_vector("woman")
)

In [14]:
scores = []

for word in glove:
    sim = cosine_similarity(result_vector, get_vector(word))

    scores.append((word, sim))

scores.sort(key=lambda x: x[1], reverse=True)

In [15]:
print("Top 10 results for king - man + woman:\n")

for word, score in scores[:10]:
    print(f"{word:15s} {score:.4f}")

Top 10 results for king - man + woman:

king            0.8860
queen           0.8610
daughter        0.7685
prince          0.7641
princess        0.7513
elizabeth       0.7506
father          0.7314
kingdom         0.7296
mother          0.7280
son             0.7280


---

# Step 9: More Arithmetic Experiments

In [16]:
experiments = [
    ("queen", "woman", "man"),
    ("doctor", "man", "woman"),
    ("teacher", "man", "woman")
]

In [17]:
for a, b, c in experiments:
    print(f"\nTesting: {a} - {b} + {c}")

    vec = get_vector(a) - get_vector(b) + get_vector(c)

    scores = []

    for word in glove:
        sim = cosine_similarity(vec, get_vector(word))

        scores.append((word, sim))

    scores.sort(key=lambda x: x[1], reverse=True)

    for word, score in scores[:5]:
        print(f"{word:15s} {score:.4f}")


Testing: queen - woman + man
queen           0.8775
king            0.8589
prince          0.8052
crown           0.7755
royal           0.7463

Testing: doctor - man + woman
doctor          0.8960
nurse           0.8424
child           0.7754
woman           0.7708
mother          0.7661

Testing: teacher - man + woman
teacher         0.9089
student         0.8091
nurse           0.7644
parents         0.7528
child           0.7384


---

# Step 10: Average Vector of a Small Sentence

We combine word vectors by averaging.

In [18]:
def sentence_vector(sentence):
    words = sentence.lower().split()

    vectors = []

    for word in words:
        if word in glove:
            vectors.append(get_vector(word))

    if len(vectors) == 0:
        return np.zeros(50)

    return np.mean(vectors, axis=0)

In [19]:
sentence = "king queen man woman"

vec = sentence_vector(sentence)

print("Sentence:", sentence)

print("Vector shape:", vec.shape)

print("First 10 values:")
print(vec[:10])

Sentence: king queen man woman
Vector shape: (50,)
First 10 values:
[ 0.1517835   0.89692751 -0.65357749 -0.26922525  1.0362375   0.55341501
 -0.2672975   0.53568501 -0.1062745  -0.3030325 ]


---

# Step 11: Sentence Similarity

In [20]:
sentences = [
    "king queen",
    "man woman",
    "doctor nurse",
    "banana orange"
]

In [21]:
base_sentence = "king man"

base_vec = sentence_vector(base_sentence)

print("Base sentence:", base_sentence)

for s in sentences:
    vec = sentence_vector(s)

    sim = cosine_similarity(base_vec, vec)

    print(f"{s:15s} -> {sim:.4f}")

Base sentence: king man
king queen      -> 0.8665
man woman       -> 0.8267
doctor nurse    -> 0.5880
banana orange   -> 0.3421


---

# Step 12: Out-of-Vocabulary Check

In [22]:
test_words = [
    "king",
    "dragon",
    "teacher",
    "spaceship"
]

In [23]:
for word in test_words:
    if word in glove:
        print(f"{word:10s} -> Found")
    else:
        print(f"{word:10s} -> Missing")

king       -> Found
dragon     -> Missing
teacher    -> Found
spaceship  -> Missing


---

# Step 13: Build a Mini Similarity Table

In [24]:
words = ["king", "queen", "man", "woman"]

In [25]:
print("Cosine Similarity Table:\n")

for w1 in words:
    row = []

    for w2 in words:
        sim = cosine_similarity(
            get_vector(w1),
            get_vector(w2)
        )

        row.append(f"{sim:.3f}")

    print(w1.ljust(8), row)

Cosine Similarity Table:

king     ['1.000', '0.784', '0.531', '0.411']
queen    ['0.784', '1.000', '0.537', '0.600']
man      ['0.531', '0.537', '1.000', '0.886']
woman    ['0.411', '0.600', '0.886', '1.000']


---

# Step 14: Final Exploration

In [26]:
custom_words = [
    "science",
    "technology",
    "teacher",
    "student"
]

In [27]:
for word in custom_words:
    if word in glove:
        print(f"\nTop neighbors for {word}:")
        
        for neighbor, score in nearest_words(word, top_n=5):
            print(f"{neighbor:15s} {score:.4f}")
    else:
        print(f"\n{word} not found in vocabulary.")


Top neighbors for science:
sciences        0.8548
research        0.8437
institute       0.8386
studies         0.8369
scientific      0.8285

Top neighbors for technology:
technologies    0.8928
computer        0.8526
systems         0.8289
software        0.8090
business        0.7864

Top neighbors for teacher:
student         0.8962
graduate        0.8133
teaching        0.8129
taught          0.8076
school          0.7855

Top neighbors for student:
teacher         0.8962
students        0.8824
teachers        0.8566
graduate        0.8301
school          0.8239
